# 2. Feature Engineering

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../resources/full_dataset.csv')
df.head()

,V020,S111A,V501,HV104,SB267,SB236,SB240,WBP24,WBP25,WBP16,...,V744A,V744B,V744C,V744D,V744E,CASEID,V001,V005,V021,V022
0,1,1,1,2,139.0,0.0,NaN,129.0,97.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1 1 3,1,156207,1,1
1,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 4 2,1,156207,1,1
2,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1 7 3,1,156207,1,1
3,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 10 2,1,156207,1,1
4,1,1,1,2,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1 13 2,1,156207,1,1


In [3]:
from tools.recode import recode, clean_recode, parse_recode, parse_all_recodes
from tools.recode import binary_cols, nominal_cols, ordinal_cols, numerical_continuous_cols, numerical_discrete_cols
from tools.recode import domain_groups

parse_all_recodes()

Code,Category Label
1,Ever-married
0,All woman
Code,Category Label
1,Currently married
2,Separated
3,Deserted
4,Divorced
5,Widowed
Code,Category Label
0,Never married


In [4]:
def parse(var: str) -> None:
    """
    Details of a variable (displayed simply)
    """
    var_dict = recode.get(var, None) or clean_recode.get(var, None)
    if not var_dict:
        print("Variable not found in the recode list!")
        return None

    print(f"Name: {var_dict['name']}")
    value = var_dict.get("value", None)
    if value is None:
        print("Value: Continuous variable")
        return None
    print(f"Categories: ")
    for i, j in value.items():
        print(f"\t{i}: {j}")

## Data Cleaning

#### Selection of currently married Women

In [5]:
parse("S111A")
print("-"*40,"\n")
print("Frequencies:")
df["S111A"].value_counts()

Name: Current marital status
Categories: 
	1: Currently married
	2: Separated
	3: Deserted
	4: Divorced
	5: Widowed
---------------------------------------- 

Frequencies:


S111A
1    28537
5      866
4      324
2      262
3       89
Name: count, dtype: int64

In [6]:
parse("HV104")
print("-"*40,"\n")
print("Frequencies:")
df["HV104"].value_counts()

Name: Sex of household member
Categories: 
	1: Male
	2: Female
	9: Missing
---------------------------------------- 

Frequencies:


HV104
2    30078
Name: count, dtype: int64

Since the entire dataset only represents females, we do not need to filter it by gender. Instead, we can just filter by `Current marital status`.

In [7]:
# Filter by "Current marital status"
df = df[df["S111A"] == 1]
df.shape

(28537, 68)

#### Exploration of `Exposure`, `Outcome` and `Effect modifier`

In [8]:
# Check values of Diabetes
for i in domain_groups.get("diabetes"):
    display(df[i].value_counts().sort_index())
    print("-"*40)

SB267
35.0       1
49.0       1
52.0       2
53.0       1
54.0       2
        ... 
346.0      1
353.0      1
357.0      1
994.0    121
996.0     42
Name: count, Length: 196, dtype: int64

----------------------------------------


SB236
0.0    4676
1.0     216
9.0      49
Name: count, dtype: int64

----------------------------------------


SB240
0.0     66
1.0    150
Name: count, dtype: int64

----------------------------------------


In [9]:
# Check values of Diabetes
for i in domain_groups.get("diabetes"):
    parse(i)
    print()
    print("-"*40)

Name: Plasma glucose (mg/dL)
Value: Continuous variable

----------------------------------------
Name: Ever diagnosed with diabetes
Categories: 
	0: No
	1: Yes
	9: Missing

----------------------------------------
Name: Currently taking medication for diabetes
Categories: 
	0: No
	1: Yes
	9: Missing

----------------------------------------


In [10]:
# Replace missing values with NaN
df["SB267"] = df["SB267"].replace([994, 996], np.nan)
df["SB236"] = df["SB236"].replace(9, np.nan)

In [11]:
# Check values of Hypertension
for i in domain_groups.get("hypertension"):
    display(df[i].value_counts().sort_index())
    print("-"*40)

WBP24
75.0     2
78.0     3
80.0     5
81.0     2
82.0     3
        ..
209.0    1
212.0    1
219.0    1
234.0    1
239.0    1
Name: count, Length: 116, dtype: int64

----------------------------------------


WBP25
47.0     2
48.0     1
49.0     3
51.0     4
52.0     7
        ..
124.0    1
125.0    3
126.0    1
127.0    1
135.0    1
Name: count, Length: 77, dtype: int64

----------------------------------------


WBP16
0.0    4396
1.0     495
Name: count, dtype: int64

----------------------------------------


WBP19
0.0    175
1.0    320
Name: count, dtype: int64

----------------------------------------


In [12]:
# Check values of Hypertension
for i in domain_groups.get("hypertension"):
    parse(i)
    print("-"*40)

Name: Final systolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Final diastolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Previously diagnosed with hypertension
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Currently taking blood pressure medication
Categories: 
	0: No
	1: Yes
----------------------------------------


In [13]:
# Check values of Obesity
for i in domain_groups.get("obesity"):
    display(df[i].value_counts().sort_index())
    print("-"*40)

HA40
1254.0      1
1300.0      1
1305.0      1
1307.0      1
1316.0      1
         ... 
4521.0      1
4528.0      1
4674.0      1
4690.0      1
9999.0    103
Name: count, Length: 1914, dtype: int64

----------------------------------------


In [14]:
# Check values of Obesity
for i in domain_groups.get("obesity"):
    parse(i)
    print("-"*40)

Name: Body Mass Index (BMI)
Value: Continuous variable
----------------------------------------


In [15]:
# Replace missing values with NaN
df["HA40"] = df["HA40"].replace(9999, np.nan)

In [16]:
# Check values of Wealth
for i in domain_groups.get("ses_wealth"):
    display(df[i].value_counts().sort_index())
    print("-"*40)

V190
1    5214
2    5553
3    5618
4    5856
5    6296
Name: count, dtype: int64

----------------------------------------


V190A
1    5730
2    5698
3    5648
4    5692
5    5769
Name: count, dtype: int64

----------------------------------------


V191
-189160    1
-187614    1
-185856    1
-184276    1
-181801    1
          ..
 345672    1
 350769    1
 361319    1
 362848    1
 370243    1
Name: count, Length: 21090, dtype: int64

----------------------------------------


V191A
-235835    1
-233964    1
-231393    1
-230310    1
-226976    1
          ..
 351396    1
 355551    1
 367549    3
 377075    1
 387535    1
Name: count, Length: 21141, dtype: int64

----------------------------------------


In [17]:
# Check values of Wealth
for i in domain_groups.get("ses_wealth"):
    parse(i)
    print("-"*40)

Name: Wealth index combined (categorical)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest
----------------------------------------
Name: Wealth index combined (categorical, urban/rural clustered)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest
----------------------------------------
Name: Wealth index factor score combined
Value: Continuous variable
----------------------------------------
Name: Wealth index factor score combined (urban/rural clustered)
Value: Continuous variable
----------------------------------------


In [18]:
# Check values of Mental health variable
for i in domain_groups.get("mental_health"):
    display(df[i].value_counts())
    print("-"*40)

MTH22
0.0     4358
2.0     2463
1.0     2390
3.0     2374
4.0     1999
5.0     1574
6.0     1099
7.0      761
8.0      553
9.0      501
10.0     232
11.0     175
12.0     111
13.0      94
14.0      71
15.0      63
16.0      43
18.0      36
17.0      31
19.0      22
21.0      13
22.0       8
20.0       5
23.0       4
27.0       2
25.0       2
24.0       2
26.0       1
Name: count, dtype: int64

----------------------------------------


MTH24
0.0     4462
1.0     2922
2.0     2734
3.0     2299
4.0     1664
5.0     1349
6.0     1127
7.0      890
8.0      473
9.0      287
10.0     199
11.0     166
12.0     108
14.0      73
13.0      62
15.0      59
16.0      35
17.0      30
18.0      16
19.0      14
21.0      11
20.0       7
Name: count, dtype: int64

----------------------------------------


In [19]:
print(domain_groups.get("diabetes"))
print(domain_groups.get("hypertension"))
print(domain_groups.get("obesity"))
print(domain_groups.get("ses_wealth"))
print(domain_groups.get("mental_health"))

['SB267', 'SB236', 'SB240']
['WBP24', 'WBP25', 'WBP16', 'WBP19']
['HA40']
['V190', 'V190A', 'V191', 'V191A']
['MTH22', 'MTH24']


##### Missing values

In [20]:
print("Missing vlaue for Diabetes")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Hypertension")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("hypertension")].isna().sum(axis=1) < 4].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Obesity")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("obesity")].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Socioeconomic Status")
print("\tMissing:", df.shape[0] - df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0])
print("\tAvailable:", df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0])
print("\tMissing:", 100 - df.iloc[df[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Depression")
print("\tMissing:", df.shape[0] - df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[["MTH22"]].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")
print("\nMissing vlaue for Anxiety")
print("\tMissing:", df.shape[0] - df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0])
print("\tAvailable:", df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0])
print("\tMissing:", 100 - df.iloc[df[["MTH24"]].isna().sum(axis=1) < 1].shape[0] * 100 / df.shape[0], "%")

Missing vlaue for Diabetes
	Missing: 23639
	Available: 4898
	Missing: 82.8363177629043 %

Missing vlaue for Hypertension
	Missing: 23646
	Available: 4891
	Missing: 82.86084732102184 %

Missing vlaue for Obesity
	Missing: 19086
	Available: 9451
	Missing: 66.88159231874408 %

Missing vlaue for Socioeconomic Status
	Missing: 0
	Available: 28537
	Missing: 0.0 %

Missing vlaue for Depression
	Missing: 9550
	Available: 18987
	Missing: 33.46532571748958 %

Missing vlaue for Anxiety


	Missing: 9550


	Available: 18987
	Missing: 33.46532571748958 %


#### Filter out missing values `Exposure`, `Outcome` and `Effect modifier`

In [21]:
# Filter missing values for Cardiometabolic Burden
df_new = df.iloc[df[domain_groups.get("diabetes")].isna().sum(axis=1) < 3]
df_new = df_new.iloc[df_new[domain_groups.get("hypertension")].isna().sum(axis=1) < 4]
df_new = df_new.iloc[df_new[domain_groups.get("obesity")].isna().sum(axis=1) < 1]

# Filter missing values for SES
df_new = df_new.iloc[df_new[domain_groups.get("ses_wealth")].isna().sum(axis=1) < 4]

# Filter missing vlaues for Depression and Anxiety
df_depression = df_new.iloc[df_new[["MTH22"]].isna().sum(axis=1) < 1]
df_anxiety = df_new.iloc[df_new[["MTH24"]].isna().sum(axis=1) < 1]

print(df_depression.shape, df_anxiety.shape)

(4887, 68) (4887, 68)


In [22]:
df_depression.equals(df_anxiety)

True

Hence both dataframes are identical

# Recategorization of all variables

## Cardiometabolic Burden

#### Diabetes Variable

In [23]:
for i in domain_groups.get("diabetes"):
    parse(i)
    print("-"*40)

Name: Plasma glucose (mg/dL)
Value: Continuous variable
----------------------------------------
Name: Ever diagnosed with diabetes
Categories: 
	0: No
	1: Yes
	9: Missing
----------------------------------------
Name: Currently taking medication for diabetes
Categories: 
	0: No
	1: Yes
	9: Missing
----------------------------------------


In [24]:
df_anxiety["Diabetes"] = ((df_anxiety["SB267"] >= 126) | (df_anxiety["SB236"] == 1) | (df_anxiety["SB240"] == 1)).astype(int)
df_anxiety["Diabetes"].value_counts()

Diabetes
0    4503
1     384
Name: count, dtype: int64

#### Hypertension Variable

In [25]:
for i in domain_groups.get("hypertension"):
    parse(i)
    print("-"*40)

Name: Final systolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Final diastolic blood pressure (mmHg)
Value: Continuous variable
----------------------------------------
Name: Previously diagnosed with hypertension
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Currently taking blood pressure medication
Categories: 
	0: No
	1: Yes
----------------------------------------


In [26]:
df_anxiety["Hypertension"] = ((df_anxiety["WBP24"] >= 140) | (df_anxiety["WBP25"] >= 90) | (df_anxiety["WBP16"] == 1) | (df_anxiety["WBP19"] == 1)).astype(int)
df_anxiety["Hypertension"].value_counts()

Hypertension
0    4013
1     874
Name: count, dtype: int64

#### BMI Variable

In [27]:
for i in domain_groups.get("obesity"):
    parse(i)
    print("-"*40)

Name: Body Mass Index (BMI)
Value: Continuous variable
----------------------------------------


In [28]:
df_anxiety["Obesity"] = (df_anxiety["HA40"] >= 3000).astype(int)
df_anxiety["Obesity"].value_counts()

Obesity
0    4477
1     410
Name: count, dtype: int64

#### Combine Cardiometabolic Burden

In [29]:
print("After recategorization:\n")
parse("Cardiometabolic Burden")

After recategorization:

Name: Cardiometabolic Burden
Categories: 
	0: None
	1: One burden
	2: Two burdens
	3: Three burdens


In [30]:
df_da = pd.DataFrame()
df_da["Cardiometabolic Burden"] = df_anxiety[["Diabetes", "Hypertension", "Obesity"]].sum(axis=1)
df_da["Cardiometabolic Burden"].value_counts()

Cardiometabolic Burden
0    3571
1    1000
2     280
3      36
Name: count, dtype: int64

In [31]:
parse("Diabetes")
print("-"*40)
parse("Hypertension")
print("-"*40)
parse("Obesity")
print("-"*40)
parse("Cardiometabolic Burden")
print("-"*40)
parse("Cardiometabolic Burden Merged")
print("-"*40)
parse("Cardiometabolic Burden Binary")

Name: Has Diabetes
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Has Hypertension
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Has Obesity
Categories: 
	0: No
	1: Yes
----------------------------------------
Name: Cardiometabolic Burden
Categories: 
	0: None
	1: One burden
	2: Two burdens
	3: Three burdens
----------------------------------------
Name: Cardiometabolic Burden
Categories: 
	0: None
	1: One burden
	2: Two or more burdens
----------------------------------------
Name: Cardiometabolic Burden
Categories: 
	0: No
	1: Yes


In [ ]:
df_da["Diabetes"] = df_anxiety["Diabetes"]
df_da["Hypertension"] = df_anxiety["Hypertension"]
df_da["Obesity"] = df_anxiety["Obesity"]
df_da["Cardiometabolic Burden Merged"] = df_da["Cardiometabolic Burden"].replace({3: 2})
df_da["Cardiometabolic Burden Binary"] = df_da["Cardiometabolic Burden Merged"].replace({2: 1})

In [33]:
display(df_da["Diabetes"].value_counts())
print("-"*40)
display(df_da["Hypertension"].value_counts())
print("-"*40)
display(df_da["Obesity"].value_counts())
print("-"*40)
display(df_da["Cardiometabolic Burden"].value_counts())
print("-"*40)
display(df_da["Cardiometabolic Burden Merged"].value_counts())
print("-"*40)
display(df_da["Cardiometabolic Burden Binary"].value_counts())

Diabetes
0    4503
1     384
Name: count, dtype: int64

----------------------------------------


Hypertension
0    4013
1     874
Name: count, dtype: int64

----------------------------------------


Obesity
0    4477
1     410
Name: count, dtype: int64

...

Cardiometabolic Burden
0    3571
1    1000
2     280
3      36
Name: count, dtype: int64

----------------------------------------


Cardiometabolic Burden Merged
0    3571
1    1000
2     316
Name: count, dtype: int64

----------------------------------------


Cardiometabolic Burden Binary
0    3571
1    1316
Name: count, dtype: int64

## Socioeconomic Status

In [34]:
print("After recategorization:\n")
parse("Socioeconomic Status")

After recategorization:

Name: Socioeconomic Status (Wealth index combined)
Categories: 
	1: Poorest
	2: Poorer
	3: Middle
	4: Richer
	5: Richest


In [35]:
df_da["Socioeconomic Status"] = df_anxiety[["V190"]]
df_da["Socioeconomic Status"].value_counts()

Socioeconomic Status
5    1124
4    1016
3     944
2     944
1     859
Name: count, dtype: int64

## Anxiety and Depression

In [36]:
print("After recategorization:\n")
parse("Depression")
print("-"*40)
parse("Anxiety")

After recategorization:

Name: PHQ-9 depression score (categorized)
Categories: 
	0: 0-4 (minimal)
	1: 5-9 (mild)
	2: 10-14 (moderate)
	3: 15-19 (moderately severe)
	4: 20-27 (severe)
----------------------------------------
Name: GAD-7 anxiety score (categorized)
Categories: 
	0: 0-4 (minimal)
	1: 5-9 (mild)
	2: 10-14 (moderate)
	3: 15-21 (severe)


In [37]:
def phq9(value: int) -> int:
    if value < 5:
        return 0
    elif value < 10:
        return 1
    elif value < 15:
        return 2
    elif value < 20:
        return 3
    else:
        return 4

def gad7(value: int) -> int:
    if value < 5:
        return 0
    elif value < 10:
        return 1
    elif value < 15:
        return 2
    else:
        return 3

def phq9_binary(value: int) -> int:
    if value < 10:
        return 0
    else:
        return 1

def gad7_binary(value: int) -> int:
    if value < 10:
        return 0
    else:
        return 1

In [38]:
df_da["Depression"] = df_anxiety["MTH22"].apply(phq9)
df_da["Anxiety"] = df_anxiety["MTH24"].apply(gad7)

df_da["Depression Binary"] = df_anxiety["MTH22"].apply(phq9_binary)
df_da["Anxiety Binary"] = df_anxiety["MTH24"].apply(gad7_binary)

display(df_da["Depression"].value_counts())
print("-"*40)
display(df_da["Anxiety"].value_counts())
print("-"*40)
display(df_da["Depression Binary"].value_counts())
print("-"*40)
df_da["Anxiety Binary"].value_counts()

Depression
0    3475
1    1177
2     173
3      57
4       5
Name: count, dtype: int64

----------------------------------------


Anxiety
0    3544
1    1122
2     175
3      46
Name: count, dtype: int64

----------------------------------------


Depression Binary
0    4652
1     235
Name: count, dtype: int64

----------------------------------------


Anxiety Binary
0    4666
1     221
Name: count, dtype: int64

In [39]:
display(df_da.head())

,Cardiometabolic Burden,Diabetes,Hypertension,Obesity,Cardiometabolic Burden Merged,Cardiometabolic Burden Binary,Socioeconomic Status,Depression,Anxiety,Depression Binary,Anxiety Binary
0,3,1,1,1,2,1,5,1,1,0,0
5,0,0,0,0,0,0,4,1,1,0,0
11,0,0,0,0,0,0,4,0,0,0,0
18,0,0,0,0,0,0,4,0,0,0,0
25,0,0,0,0,0,0,5,0,0,0,0


## Recategorization of Confounders

#### Education

In [40]:
parse("V106")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Education")

Name: Highest educational level
Categories: 
	0: No education
	1: Primary
	2: Secondary
	3: Higher
---------------------------------------- 

After recategorization:

Name: Highest educational level
Categories: 
	0: No education
	1: Primary
	2: Secondary
	3: Higher


In [41]:
print("Available categories in the dataset")
print(df["V106"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3]


In [42]:
df_da["Education"] = df_anxiety["V106"]
df_da["Education"].value_counts()

Education
2    2172
1    1276
3     780
0     659
Name: count, dtype: int64

#### Occupation

In [43]:
parse("V714")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Occupation")

Name: Respondent currently working
Categories: 
	0: No
	1: Yes
---------------------------------------- 

After recategorization:

Name: Respondent currently working
Categories: 
	0: No
	1: Yes


In [44]:
print("Available categories in the dataset")
print(df["V714"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0]


In [45]:
df_da["Occupation"] = df_anxiety["V714"]
df_da["Occupation"].value_counts()

Occupation
0.0    3369
1.0    1518
Name: count, dtype: int64

#### Husband/partner's occupation (grouped)

In [46]:
parse("V705")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Partner occupation")

Name: Husband/partner's occupation (grouped)
Categories: 
	0: Not working
	1: Professional/technical/managerial
	2: Clerical
	3: Sales
	4: Agricultural - self employed
	5: Agricultural - employee
	6: Household and domestic
	7: Services
	8: Skilled manual
	9: Unskilled manual
	98: Don't know
---------------------------------------- 

After recategorization:

Name: Husband/partner's occupation (grouped)
Categories: 
	1: Not working
	2: Working
	3: Don't know


In [47]:
print("Available categories in the dataset")
print(df["V705"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 98.0]


In [48]:
def po(value: int) -> int:
    if value == 0:
        return 1          # Not working
    elif value == 98:
        return 3          # Don't know
    elif value < 10:
        return 2          # Working
    else:
        return np.nan      # Missing

In [49]:
df_da["Partner occupation"] = df_anxiety["V705"].apply(po)
df_da["Partner occupation"].value_counts()

Partner occupation
2    4734
1     145
3       8
Name: count, dtype: int64

#### Age group

In [50]:
parse("V013")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Age")

Name: Age in 5-year groups
Categories: 
	1: 15-19
	2: 20-24
	3: 25-29
	4: 30-34
	5: 35-39
	6: 40-44
	7: 45-49
---------------------------------------- 

After recategorization:

Name: Age in 5-year groups
Categories: 
	1: 15-24
	2: 25-34
	3: 35-49


In [51]:
print("Available categories in the dataset")
print(df["V013"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7]


In [52]:
def age(value: int) -> int:
    if value in [1, 2]:
        return 1
    elif value in [3, 4]:
        return 2
    elif value < 8:
        return 3
    else:
        return np.nan

In [53]:
df_da["Age"] = df_anxiety["V013"].apply(age)
df_da["Age"].value_counts()

Age
3    2028
2    1737
1    1122
Name: count, dtype: int64

#### Age at first cohabitation

In [54]:
parse("V511")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Age at first cohabitation")

Name: Age at first cohabitation
Value: Continuous variable
---------------------------------------- 

After recategorization:

Name: Age at first cohabitation, grouped
Categories: 
	0: <15
	1: 15-24
	2: 25-34
	3: 35-49


In [55]:
def cohabitation(value: int):
    if value < 15:
        return 0
    elif value < 25:
        return 1
    elif value < 35:
        return 2
    elif value >= 35:
        return 3
    else:
        return np.nan

In [56]:
df_da["Age at first cohabitation"] = df_anxiety["V511"].apply(cohabitation)
df_da["Age at first cohabitation"].value_counts()

Age at first cohabitation
1    3334
0    1351
2     193
3       9
Name: count, dtype: int64

#### Residence (Division)

In [57]:
parse("V024")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Division")

Name: Division
Categories: 
	1: Barishal
	2: Chattogram
	3: Dhaka
	4: Khulna
	5: Mymensingh
	6: Rajshahi
	7: Rangpur
	8: Sylhet
---------------------------------------- 

After recategorization:

Name: Division
Categories: 
	1: Barishal
	2: Chattogram
	3: Dhaka
	4: Khulna
	5: Mymensingh
	6: Rajshahi
	7: Rangpur
	8: Sylhet


In [58]:
print("Available categories in the dataset")
print(df["V024"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7, 8]


In [59]:
df_da["Division"] = df_anxiety["V024"]
df_da["Division"].value_counts()

Division
2    736
3    700
4    628
6    621
7    586
1    548
8    535
5    533
Name: count, dtype: int64

#### Residence (Urban/Rural)

In [60]:
parse("V025")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Residence")

Name: Type of place of residence
Categories: 
	1: Urban
	2: Rural
---------------------------------------- 

After recategorization:

Name: Type of place of residence
Categories: 
	1: Urban
	2: Rural


In [61]:
print("Available categories in the dataset")
print(df["V025"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2]


In [62]:
df_da["Residence"] = df_anxiety["V025"]
df_da["Residence"].value_counts()

Residence
2    3178
1    1709
Name: count, dtype: int64

#### Religion

In [63]:
parse("V130")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Religion")

Name: Religion


Categories: 
	1: Islam
	2: Hindu
	3: Buddhist
	4: Christianity
	96: Others
---------------------------------------- 

After recategorization:

Name: Religion
Categories: 
	1: Islam
	2: Others


In [64]:
print("Available categories in the dataset")
print(df["V130"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 96]


In [65]:
def religion(value: int) -> int:
    if value == 1:
        return 1
    else:
        return 2

In [66]:
df_da["Religion"] = df_anxiety["V130"].apply(religion)
df_da["Religion"].value_counts()

Religion
1    4391
2     496
Name: count, dtype: int64

#### Total children ever born

In [67]:
parse("V201")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Children")

Name: Total children ever born
Value: Continuous variable
---------------------------------------- 

After recategorization:

Name: Total children ever born
Categories: 
	0: No children
	1: 1
	2: 2
	3: 3
	4: 4 or more


In [68]:
print("Available categories in the dataset")
print(df["V201"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [69]:
def children(value: int) -> int:
    if value in range(0, 4):
        return value
    elif value < 12:
        return 4
    else:
        return np.nan

In [70]:
df_da["Children"] = df_anxiety["V201"].apply(children)
df_da["Children"].value_counts()

Children
2    1594
1    1066
3    1038
4     771
0     418
Name: count, dtype: int64

#### Number of household members

In [71]:
parse("V136")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Family size")

Name: Number of household members
Value: Continuous variable
---------------------------------------- 

After recategorization:

Name: Number of household members
Categories: 
	1: Less than 5
	2: 5 or more


In [72]:
print("Available categories in the dataset")
print(df["V136"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25]


In [73]:
def size(value: int) -> int:
    if value in range(0, 5):
        return 1
    elif value < 26:
        return 2
    else:
        return np.nan

In [74]:
df_da["Family size"] = df_anxiety["V136"].apply(size)
df_da["Family size"].value_counts()

Family size
2    2610
1    2277
Name: count, dtype: int64

#### Household Autonomy (decision-making: health care, purchases, family visits)

In [75]:
display(df_anxiety[domain_groups.get("autonomy")].isna().sum())

V743A    0
V743B    0
V743D    0
dtype: int64

In [76]:
for i in domain_groups.get("autonomy"):
    parse(i)

print("-"*40,"\n")
print("After recategorization:\n")
parse("Household Autonomy")

Name: Person who decides on respondent's health care
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
Name: Person who decides on large household purchases
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
Name: Person who decides on visits to family/relatives
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	6: Other
	9: Missing
---------------------------------------- 

After recategorization:

Name: Autonomy in household decisions (health care, purchases, family visits)
Categories: 
	0: No autonomy
	1: 1 decision
	2: 2 decisions
	3: 3 decisions


In [77]:
print("Available categories in the dataset")
for i in domain_groups.get("autonomy"):
    print(df[i].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1.0, 2.0, 4.0, 5.0, 6.0]
[1.0, 2.0, 4.0, 5.0, 6.0]
[1.0, 2.0, 4.0, 5.0, 6.0]


In [78]:
def autonomy(value: int) -> int:
    if value in [1, 2, 3]:
        return 1
    elif value in [4, 5, 6]:
        return 0
    else:
        return np.nan

In [79]:
df_da["Household Autonomy"] = df_anxiety[domain_groups.get("autonomy")].map(autonomy).sum(axis=1)
df_da["Household Autonomy"].value_counts()

Household Autonomy
3    2874
0     746
2     674
1     593
Name: count, dtype: int64

#### Decision-making over money husband earns (kept as a separate variable)

`V743F` ("who decides what to do with money husband earns") is **not** folded into `Household Autonomy` above. Unlike the other three decision items, `V743F` carries a structural response category (`7` = "Husband/partner has no earnings") that doesn't exist for `V743A`/`V743B`/`V743D` — it isn't a lower level of autonomy, it's a different question not applying to households with no cash income for the husband to control. Merging it in would force an arbitrary denominator correction (some women's scores out of 3 items, others out of 4) or an unjustified assumption that "no earnings" equals "no autonomy."

This mirrors DHS's own convention: the official *Participation in Decision Making* indicator and SDG Indicator 5.6.1 are both built from exactly `V743A`, `V743B`, `V743D` — the money-earned item is tracked as a separate empowerment dimension (*Control over Women's Cash Earnings* / *Decision-Making Over Financial Assets*), never summed with the other three (DHS Guide to DHS Statistics, "Participation in Decision Making"; Kishor & Subaiya 2008, DHS Comparative Reports No. 20).

In [80]:
parse("V743F")
print("-"*40, "\n")
print("After recategorization:\n")
parse("Financial Decision-Making")

Name: Person who decides what to do with money husband earns
Categories: 
	1: Respondent alone
	2: Respondent & husband/partner jointly
	3: Respondent and other person
	4: Husband/partner alone
	5: Someone else
	7: Husband/partner has no earnings
	6: Other
	9: Missing
---------------------------------------- 

After recategorization:

Name: Person who decides what to do with money husband earns
Categories: 
	0: Husband/other decides
	1: Respondent has a say
	2: No earnings (N/A)


In [81]:
print("Available categories in the dataset")
print(df["V743F"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1.0, 2.0, 4.0, 6.0, 7.0]


In [82]:
def financial_decision(value: int) -> int:
    if value in [1, 2, 3]:
        return 1   # Respondent has a say
    elif value in [4, 5, 6]:
        return 0   # Respondent has no say
    elif value == 7:
        return 2   # No earnings - not applicable
    else:
        return np.nan

In [83]:
df_da["Financial Decision-Making"] = df_anxiety["V743F"].apply(financial_decision)
df_da["Financial Decision-Making"].value_counts()

Financial Decision-Making
1    3238
0    1585
2      64
Name: count, dtype: int64

#### Justification for Physical/Sexual/Emotional Abuse

Recoded as a 3-level variable (`Rejects` / `Uncertain` / `Justifies`) rather than binary, so that "don't know" responses get their own category instead of being silently absorbed into "does not justify." Merge rule mirrors DHS's own "any Yes wins" convention, extended with a second tier: **Justifies** if any item = Yes; else **Uncertain** if any item = Don't know; else **Rejects**.

In [84]:
display(df_anxiety[domain_groups.get("ipv_attitudes")].isna().sum())

V744A    0
V744B    0
V744C    0
V744D    0
V744E    0
dtype: int64

No missing vlaues

In [85]:
for i in domain_groups.get("ipv_attitudes"):
    parse(i)

print("-"*40,"\n")
print("After recategorization:\n")
parse("IPV Attitude")

Name: Wife beating justified if she goes out without telling husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she neglects the children
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she argues with husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she refuses to have sex with husband
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
Name: Wife beating justified if she burns the food
Categories: 
	0: No
	1: Yes
	8: Don't know
	9: Missing
---------------------------------------- 

After recategorization:

Name: Justification for Physical/Sexual/Emotional Abuse
Categories: 
	0: Rejects in all scenarios
	1: Uncertain (Don't know, never affirms)
	2: Justifies in ≥1 scenario


In [86]:
print("Available categories in the dataset")
for i in domain_groups.get("ipv_attitudes"):
    print(df[i].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]
[0.0, 1.0, 8.0]


In [87]:
def merge_ipv(row) -> float:
    valid = row.dropna()
    if len(valid) == 0:             # Missing
        return np.nan
    if (valid == 1).any():          # Justifies in >=1 scenario
        return 2
    elif (valid == 8).any():        # Never affirms, but unresolved on >=1 item
        return 1
    else:                           # Rejects in every answered scenario
        return 0

In [88]:
df_da["IPV Attitude"] = df_anxiety[domain_groups.get("ipv_attitudes")].apply(merge_ipv, axis=1)
df_da["IPV Attitude"].value_counts()

IPV Attitude
0    4174
2     651
1      62
Name: count, dtype: int64

#### Health insurance

In [89]:
parse("V481")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Insurance")

Name: Covered by health insurance
Categories: 
	0: No
	1: Yes
	9: Missing
---------------------------------------- 

After recategorization:

Name: Covered by health insurance
Categories: 
	0: No
	1: Yes


In [90]:
print("Available categories in the dataset")
print(df["V481"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0]


In [91]:
def insurance(value: int) -> int:
    if value in [0, 1]:
        return value
    else:
        return np.nan

In [92]:
df_da["Insurance"] = df_anxiety["V481"].apply(insurance)
df_da["Insurance"].value_counts()

Insurance
0.0    4871
1.0      16
Name: count, dtype: int64

#### Mass media use (TV and Internet)

In [93]:
parse("V171A")
print("-"*40,"\n")
parse("V159")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Mass Media")

Name: Use of internet
Categories: 
	0: Never
	1: Yes, last 12 months
	2: Yes, before last 12 months
	3: Yes, can't establish when
	9: Missing
---------------------------------------- 

Name: Frequency of watching television
Categories: 
	0: Not at all
	1: Less than once a week
	2: At least once a week
	3: Almost every day
	9: Missing
---------------------------------------- 

After recategorization:

Name: Use of mass media
Categories: 
	0: Never
	1: Only television
	2: Only internet
	3: Both television and internet


In [94]:
conditions = [
    (df_anxiety["V171A"] == 0) & (df_anxiety["V159"] == 0),
    (df_anxiety["V159"].isin([1, 2, 3])) & (df_anxiety["V171A"] == 0),
    (df_anxiety["V171A"].isin([1, 2, 3])) & (df_anxiety["V159"] == 0),
    (df_anxiety["V159"].isin([1, 2, 3])) & (df_anxiety["V171A"].isin([1, 2, 3]))
]

choices = [0, 1, 2, 3]

In [95]:
df_da["Mass Media"] = np.select(conditions, choices, default=np.nan)
df_da["Mass Media"].value_counts()

Mass Media
1.0    1808
0.0    1674
3.0     907
2.0     498
Name: count, dtype: int64

#### Current contraceptive use by method type

In [96]:
parse("V313")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Contraceptive")

Name: Current use by method type (simplified/collapsed version)
Categories: 
	0: No method
	1: Folkloric method
	2: Traditional method
	3: Modern method
	9: Missing
---------------------------------------- 

After recategorization:

Name: Current contraceptive use by method type (simplified/collapsed version)
Categories: 
	0: No method
	1: Folkloric method
	2: Traditional method
	3: Modern method


In [97]:
print("Available categories in the dataset")
print(df["V313"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0.0, 1.0, 2.0, 3.0]


In [98]:
def contraceptive(value: int) -> int:
    if value in range(0, 4):
        return value
    else:
        return np.nan

In [99]:
df_da["Contraceptive"] = df_anxiety["V313"].apply(contraceptive)
df_da["Contraceptive"].value_counts()

Contraceptive
3.0    2689
0.0    1676
2.0     517
1.0       5
Name: count, dtype: int64

#### Miscarriage/Abortion

In [100]:
parse("V228")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Abortion")

Name: Ever had a terminated pregnancy
Categories: 
	0: No
	1: Yes
	9: Missing
---------------------------------------- 

After recategorization:

Name: Ever had a terminated pregnancy
Categories: 
	0: No
	1: Yes


In [101]:
print("Available categories in the dataset")
print(df["V228"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1]


In [102]:
def abortion(value: int) -> int:
    if value in range(0, 2):
        return value
    else:
        return np.nan

In [103]:
df_da["Abortion"] = df_anxiety["V228"].apply(abortion)
df_da["Abortion"].value_counts()

Abortion
0    3616
1    1271
Name: count, dtype: int64

#### Currently pregnant

In [104]:
parse("V213")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Pregnant")

Name: Currently pregnant
Categories: 
	0: No or unsure
	1: Yes
	9: Missing
---------------------------------------- 

After recategorization:

Name: Currently pregnant
Categories: 
	0: No or unsure
	1: Yes


In [105]:
print("Available categories in the dataset")
print(df["V213"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1]


In [106]:
def pregnant(value: int) -> int:
    if value in range(0, 2):
        return value
    else:
        return np.nan

In [107]:
df_da["Pregnant"] = df_anxiety["V213"].apply(pregnant)
df_da["Pregnant"].value_counts()

Pregnant
0    4601
1     286
Name: count, dtype: int64

#### In menopause

In [108]:
parse("V226")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Menopause")

Name: Time since last period (comp) (months)
Categories: 
	Continuous: 0:400
	994: In menopause
	995: Before last pregnancy
	996: Never menstruated
	997: Inconsistent
	998: Don't know
	999: Missing
---------------------------------------- 

After recategorization:

Name: In menopause
Categories: 
	0: No
	1: Yes


In [109]:
print("Available categories in the dataset")
print(df["V226"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 27, 28, 30, 31, 34, 36, 41, 42, 45, 46, 48, 56, 58, 60, 71, 72, 84, 96, 108, 120, 132, 144, 156, 168, 180, 192, 204, 216, 228, 240, 276, 994, 995, 996, 997]


In [110]:
def menopause(value: int) -> int:
    if value == 994:
        return 1
    elif value < 998:
        return 0
    else:
        return np.nan

In [111]:
df_da["Menopause"] = df_anxiety["V226"].apply(menopause)
df_da["Menopause"].value_counts()

Menopause
0    4621
1     266
Name: count, dtype: int64

#### Recent sexual activity

In [112]:
parse("V536")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Sexual activity")

Name: Recent sexual activity
Categories: 
	0: Never had sex
	1: Active in last 4 weeks
	2: Not active in last 4 weeks - postpartum abstinence
	3: Not active in last 4 weeks - not postpartum abstinence
	9: Missing
---------------------------------------- 

After recategorization:

Name: Recent sexual activity
Categories: 
	0: Not Active
	1: Active


In [113]:
print("Available categories in the dataset")
print(df["V536"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1.0, 2.0, 3.0]


In [114]:
def sexual_activity(value: int) -> int:
    if value == 1:
        return 1
    elif value in [0, 2, 3]:
        return 0
    else:
        return np.nan

In [115]:
df_da["Sexual activity"] = df_anxiety["V536"].apply(sexual_activity)
df_da["Sexual activity"].value_counts()

Sexual activity
1    3930
0     957
Name: count, dtype: int64

#### Postpartum abstinence

In [116]:
parse("V536")
print("-"*40,"\n")
print("After recategorization:\n")
parse("Postpartum")

Name: Recent sexual activity
Categories: 
	0: Never had sex
	1: Active in last 4 weeks
	2: Not active in last 4 weeks - postpartum abstinence
	3: Not active in last 4 weeks - not postpartum abstinence
	9: Missing
---------------------------------------- 

After recategorization:

Name: Postpartum abstinence
Categories: 
	0: No
	1: Yes


In [117]:
print("Available categories in the dataset")
print(df["V536"].value_counts().index.sort_values().to_list())

Available categories in the dataset
[1.0, 2.0, 3.0]


In [118]:
def postpartum(value: int) -> int:
    if value == 2:
        return 1
    elif value in [0, 1, 3]:
        return 0
    else:
        return np.nan

In [119]:
df_da["Postpartum"] = df_anxiety["V536"].apply(postpartum)
df_da["Postpartum"].value_counts()

Postpartum
0    4770
1     117
Name: count, dtype: int64

## Sampling Design

In [120]:
(df['V001'] == df['V021']).mean()

np.float64(1.0)

Since the mean is 1, hence `V001` (`Cluster number`) and `V021` (`PSU`) are identical. We can use `V021` instead of `V001`

In [ ]:
df_da["CASEID"] = df_anxiety["CASEID"]
# df_da["Cluster number"] = df_anxiety["V001"]
df_da["Sampling weight"] = df_anxiety["V005"] / 1000000
df_da["PSU"] = df_anxiety["V021"]
df_da["Stratum"] = df_anxiety["V022"]

In [122]:
df_da.to_csv('../resources/depression-anxiety-dataset.csv', index=False)